*0.2 Math / ML basics*

# PCA

**The situation.** Storage: 5 million chunks × 1,536 numbers × 4 bytes = 31 GB of vectors in memory, and the bill grows with the corpus. Most of those 1,536 directions carry very little; a few carry almost everything. Also, a data scientist wants to *look* at the embedding space from the previous item.

**PCA (principal component analysis).** Find the directions along which the points spread out the most, and keep only those. Direction 1 captures the most variation, direction 2 the next most, and so on. Keep 2 for a picture; keep 256 for a smaller index that ranks almost the same as the full one. It is a matrix multiplication, fast and exact.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Squash the 12 words to 2 numbers each.** Fit PCA, look at how much of the spread the two directions keep, and print the coordinates.

In [2]:
import numpy as np
from openai import OpenAI
from sklearn.decomposition import PCA

client = OpenAI(timeout=30)
words = [
    "delivery",
    "shipping",
    "courier",
    "tracking number",
    "invoice",
    "receipt",
    "refund",
    "payment",
    "password",
    "login",
    "two-factor code",
    "username",
]
vectors = []
for item in client.embeddings.create(model="text-embedding-3-small", input=words).data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)

pca = PCA(n_components=2, random_state=0)
points = pca.fit_transform(vectors)
print(
    "variation kept by 2 directions:",
    np.round(pca.explained_variance_ratio_, 3),
    "→ total",
    f"{pca.explained_variance_ratio_.sum():.0%}",
)
for word, (x, y) in zip(words, points):
    print(f"{word:<16} x={x:>6.2f}  y={y:>6.2f}")
within = []
between = []
for a in range(12):
    for b in range(a + 1, 12):
        distance = float(np.linalg.norm(points[a] - points[b]))
        if a // 4 == b // 4:
            within.append(distance)  # same group of four
        else:
            between.append(distance)
print(
    "average distance inside a group:",
    round(np.mean(within), 2),
    "| between groups:",
    round(np.mean(between), 2),
)
assert np.mean(within) < np.mean(between)

variation kept by 2 directions: [0.227 0.159] → total 39%
delivery         x= -0.46  y=  0.20
shipping         x= -0.40  y=  0.19
courier          x= -0.27  y=  0.40
tracking number  x= -0.13  y=  0.45
invoice          x= -0.24  y= -0.34
receipt          x= -0.09  y= -0.28
refund           x= -0.18  y= -0.42
payment          x= -0.20  y= -0.34
password         x=  0.58  y= -0.04
login            x=  0.36  y= -0.17
two-factor code  x=  0.47  y=  0.39
username         x=  0.55  y= -0.03
average distance inside a group: 0.23 | between groups: 0.79


**Reading the output.** Two numbers per word instead of 1,536, and the three groups sit apart: the four delivery words share a region, the four billing words another, the four account words a third. The variation kept says how much of the original spread survives — two directions keep a meaningful slice of twelve carefully chosen words; for a real corpus it would be far less, which is why two dimensions are for pictures only.

**The storage use: keep 256 directions and check the ranking survives.** Random unit vectors stand in for a corpus; the check is what matters.

In [3]:
from sklearn.preprocessing import normalize

rng = np.random.default_rng(0)
corpus = normalize(rng.standard_normal((5_000, 1536)).astype(np.float32))
query = corpus[:1] + 0.1 * rng.standard_normal((1, 1536)).astype(np.float32)  # near document 0

reducer = PCA(n_components=256, random_state=0).fit(corpus)
small_corpus = normalize(reducer.transform(corpus))
small_query = normalize(reducer.transform(query))

top_full = int(np.argmax(corpus @ query.T))
top_small = int(np.argmax(small_corpus @ small_query.T))
print("top result, 1536 dims:", top_full, "| 256 dims:", top_small, "| memory: 6× smaller")
assert top_full == top_small

top result, 1536 dims: 0 | 256 dims: 0 | memory: 6× smaller


**The rule to remember.** PCA keeps the directions that matter and drops the rest. Two for a picture, a few hundred for a cheaper index that ranks the same.

| Use it when | Don't when | Instead use |
|---|---|---|
| shrinking vectors for storage; a quick, faithful 2-D overview | the structure is curved and local (clusters inside clusters) — PCA flattens it | UMAP or t-SNE for pictures |

**Watch out**
- Fit PCA once on a sample and save it; the *same* transform must be applied to queries and documents forever.
- Distances in a PCA plot are honest (it is a linear projection) — unlike the next two items.
- Newer models offer a shorter output size directly (`dimensions=256` on OpenAI), which is PCA done for you during training.